### Adaboost Regression Implementation

### Used Car Price Prediction

 - 1) Problem statement.
    - This dataset comprises used cars sold on cardehko.com in India as well as important features of these cars.
    - If user can predict the price of the car based on input features.
    - Prediction results can be used to give new seller the price suggestion based on market condition.


  - 2) Data Collection.
     - The Dataset is collected from scrapping from cardheko webiste
     - The data consists of 13 column and 15411 rows.

In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [33]:
df = pd.read_csv("cardekho_imputated.csv", index_col=[0])

In [34]:
df

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19537,Hyundai i10,Hyundai,i10,9,10723,Dealer,Petrol,Manual,19.81,1086,68.05,5,250000
19540,Maruti Ertiga,Maruti,Ertiga,2,18000,Dealer,Petrol,Manual,17.50,1373,91.10,7,925000
19541,Skoda Rapid,Skoda,Rapid,6,67000,Dealer,Diesel,Manual,21.14,1498,103.52,5,425000
19542,Mahindra XUV500,Mahindra,XUV500,5,3800000,Dealer,Diesel,Manual,16.00,2179,140.00,7,1225000


In [35]:
df.head()

,car_name,brand,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Maruti Alto,Maruti,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Hyundai Grand,Hyundai,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,Hyundai i20,Hyundai,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Maruti Alto,Maruti,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ford Ecosport,Ford,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


## Data Cleaning
- Handling Missing values
- Handling Duplicates
- Check data type
- Understand the dataset

In [36]:
# Check the Null Values
# check the features with nan values
df.isnull().sum()

car_name             0
brand                0
model                0
vehicle_age          0
km_driven            0
seller_type          0
fuel_type            0
transmission_type    0
mileage              0
engine               0
max_power            0
seats                0
selling_price        0
dtype: int64

In [37]:
# Remove Unnecessary Columns 
df.drop('car_name', axis=1, inplace=True)
df.drop('brand', axis=1, inplace=True)

In [38]:
df.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats,selling_price
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5,120000
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5,550000
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5,215000
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5,226000
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5,570000


In [39]:
df['model'].unique()

<ArrowStringArray>
[        'Alto',        'Grand',          'i20',     'Ecosport',
      'Wagon R',          'i10',        'Venue',        'Swift',
        'Verna',       'Duster',
 ...
     'Panamera',      'Alturas',       'Altroz',           'NX',
     'Carnival',            'C',           'RX',        'Ghost',
 'Quattroporte',       'Gurkha']
Length: 120, dtype: str

### Getting All Different Types of features

In [40]:
num_features = [ feature for feature in df.columns if df[feature].dtype != 'str']
print("Num of Numerical features : ", len(num_features))

cat_features = [ feature for feature in df.columns if df[feature].dtype == 'str']
print("Num of Categorical features :", len(cat_features))

discrete_features = [ feature for feature in num_features if len(df[feature].unique())<=25]
print("Num of Discrete features :", len(discrete_features))

continuous_feature = [ feature for feature in num_features if feature not in discrete_features]
print("Num of Continuous Features :", len(continuous_feature))

Num of Numerical features :  7
Num of Categorical features : 4
Num of Discrete features : 2
Num of Continuous Features : 5


In [41]:
# Independent and Dependent Features
from sklearn.model_selection import train_test_split
X = df.drop(['selling_price'], axis=1)
y = df['selling_price']

In [42]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,Alto,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,Grand,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,i20,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,Alto,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,Ecosport,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [43]:
y.head()

0    120000
1    550000
2    215000
3    226000
4    570000
Name: selling_price, dtype: int64

### Feature Encoding and Scaling
- One Hot Encoding for Columns which had lesser unique values and not ordinal

- One hot encoding is a process by which categorical variables are converted into a form that could be provided to ML algorithms to do a better job in prediction.

In [44]:
df['model'].unique()

<ArrowStringArray>
[        'Alto',        'Grand',          'i20',     'Ecosport',
      'Wagon R',          'i10',        'Venue',        'Swift',
        'Verna',       'Duster',
 ...
     'Panamera',      'Alturas',       'Altroz',           'NX',
     'Carnival',            'C',           'RX',        'Ghost',
 'Quattroporte',       'Gurkha']
Length: 120, dtype: str

In [45]:
len(df['model'].unique())

120

In [46]:
df['model'].value_counts()

model
i20             906
Swift Dzire     890
Swift           781
Alto            778
City            757
               ... 
Altroz            1
C                 1
Ghost             1
Quattroporte      1
Gurkha            1
Name: count, Length: 120, dtype: int64

In [47]:
from sklearn.preprocessing import  LabelEncoder 
le = LabelEncoder()
X['model'] = le.fit_transform(X['model'])

In [48]:
X.head()

,model,vehicle_age,km_driven,seller_type,fuel_type,transmission_type,mileage,engine,max_power,seats
0,7,9,120000,Individual,Petrol,Manual,19.70,796,46.30,5
1,54,5,20000,Individual,Petrol,Manual,18.90,1197,82.00,5
2,118,11,60000,Individual,Petrol,Manual,17.00,1197,80.00,5
3,7,9,37000,Individual,Petrol,Manual,20.92,998,67.10,5
4,38,6,30000,Dealer,Diesel,Manual,22.77,1498,98.59,5


In [49]:
len(df['seller_type'].unique()),len(df['fuel_type'].unique()),len(df['transmission_type'].unique())

(3, 5, 2)

### Create Column Transformer with 3 types of Transformers

In [50]:
num_features = X.select_dtypes(exclude= 'str').columns
onehot_columns = ['seller_type', 'fuel_type', 'transmission_type']

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, onehot_columns),
        ("StandardScaler", numeric_transformer, num_features)
    ] , remainder= 'passthrough'
)

In [51]:
X = preprocessor.fit_transform(X)

In [52]:
X

array([[ 1.        ,  0.        ,  0.        , ..., -1.32425883,
        -1.26335238, -0.40302241],
       [ 1.        ,  0.        ,  0.        , ..., -0.55471774,
        -0.43257082, -0.40302241],
       [ 1.        ,  0.        ,  0.        , ..., -0.55471774,
        -0.47911321, -0.40302241],
       ...,
       [ 0.        ,  0.        ,  1.        , ...,  0.02291783,
         0.06822523, -0.40302241],
       [ 0.        ,  0.        ,  1.        , ...,  1.32979434,
         0.91715831,  2.07344426],
       [ 0.        ,  0.        ,  0.        , ...,  0.02099878,
         0.39588361, -0.40302241]], shape=(15411, 14))

In [53]:
pd.DataFrame(X)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.519714,0.983562,1.247335,-0.000276,-1.324259,-1.263352,-0.403022
1,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-0.225693,-0.343933,-0.690016,-0.192071,-0.554718,-0.432571,-0.403022
2,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.536377,1.647309,0.084924,-0.647583,-0.554718,-0.479113,-0.403022
3,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-1.519714,0.983562,-0.360667,0.292211,-0.936610,-0.779312,-0.403022
4,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-0.666211,-0.012060,-0.496281,0.735736,0.022918,-0.046502,-0.403022
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15406,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.508844,0.983562,-0.869744,0.026096,-0.767733,-0.757204,-0.403022
15407,0.0,0.0,0.0,0.0,0.0,1.0,1.0,-0.556082,-1.339555,-0.728763,-0.527711,-0.216964,-0.220803,2.073444
15408,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.407551,-0.012060,0.220539,0.344954,0.022918,0.068225,-0.403022
15409,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.426247,-0.343933,72.541850,-0.887326,1.329794,0.917158,2.073444


## Model Traing and Model Selection

In [54]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [55]:
## Create a function to Evaluate Model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2_square = r2_score(true, predicted)
    return mae, rmse,r2_square 


In [56]:
# Beginning Model Training
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

models = {

    "Linear Regression" : LinearRegression(),
    "Lasso" : Lasso(),
    "Ridge" : Ridge(),
    "K-Neighbours Regression" : KNeighborsRegressor(),
    "Decision Tree" : DecisionTreeRegressor(),
    "Random Forest " : RandomForestRegressor(),
    "Adaboost Regression" : AdaBoostRegressor(),

}

for i in range(len(list(models))):
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train Model

    # Make Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    #Evaluate Train and Test Dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae ,  model_test_rmse , model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])


    # Model Performance for Training Set

    print('Model Performance for Training Set')
    print(" - Root Mean Squared Error : {:.4f}" .format(model_train_rmse))
    print(" - Mean Absolute Error: {:.4f}" .format(model_train_mae))
    print(" - R2 Score : {:.4f}" .format(model_train_r2))

    print("----------------------------------------------------------")

    # Model Performance for Test Set
    
    print('Model Performance for Test Set')
    print(" - Root Mean Squared Error : {:.4f}" .format(model_test_rmse))
    print(" - Mean Absolute Error: {:.4f}" .format(model_test_mae))
    print(" - R2 Score : {:.4f}" .format(model_test_r2))

    print('=' *35)
    print('\n')


Linear Regression
Model Performance for Training Set
 - Root Mean Squared Error : 553855.6665
 - Mean Absolute Error: 268101.6071
 - R2 Score : 0.6218
----------------------------------------------------------
Model Performance for Test Set
 - Root Mean Squared Error : 502543.5930
 - Mean Absolute Error: 279618.5794
 - R2 Score : 0.6645


Lasso
Model Performance for Training Set
 - Root Mean Squared Error : 553855.6710
 - Mean Absolute Error: 268099.2226
 - R2 Score : 0.6218
----------------------------------------------------------
Model Performance for Test Set
 - Root Mean Squared Error : 502542.6696
 - Mean Absolute Error: 279614.7461
 - R2 Score : 0.6645


Ridge
Model Performance for Training Set
 - Root Mean Squared Error : 553856.3160
 - Mean Absolute Error: 268059.8015
 - R2 Score : 0.6218
----------------------------------------------------------
Model Performance for Test Set
 - Root Mean Squared Error : 502533.8230
 - Mean Absolute Error: 279557.2169
 - R2 Score : 0.6645


K

### Initialize few parameter for hyperparameter tuning

In [60]:
### Initialize few parameter for hyperparameter tuning
knn_params = {"n_neighbors": [2, 3, 10, 20, 40, 50]}

rf_params = {"max_depth": [5, 8, 15, None, 10],
             "max_features": [5, 7, "sqrt", 8],
             "min_samples_split": [2, 8, 15, 20],
             "n_estimators": [100, 200, 500, 1000]}

ada_params ={
    "n_estimators" : [50, 60, 70, 80],
    "loss" : ['linear', 'square', 'exponential']
}

In [61]:
# Models List for Hyperparameter Tuning

randomcv_models = [( 'KNN', KNeighborsRegressor(), knn_params),
                   ("RF", RandomForestRegressor(), rf_params),
                   ("Adaboost", AdaBoostRegressor(), ada_params)
                   ]

In [62]:
# Hyperparameter Tunining
from sklearn.model_selection import RandomizedSearchCV

model_param = {}
for name, model, params in randomcv_models:
    random = RandomizedSearchCV(estimator=model,
                                param_distributions=params,
                                n_iter=100,
                                cv=3,
                                verbose=2,
                                n_jobs=1,
                                random_state=42)

    random.fit(X_train, y_train)
    model_param[name] = random.best_params_

for model_name in model_param:
    print(f"-----------------Best Params For {model_name}----------------")
    print(model_param[model_name])


Fitting 3 folds for each of 6 candidates, totalling 18 fits
[CV] END ......................................n_neighbors=2; total time=   0.1s
[CV] END ......................................n_neighbors=2; total time=   0.1s
[CV] END ......................................n_neighbors=2; total time=   0.1s
[CV] END ......................................n_neighbors=3; total time=   0.1s
[CV] END ......................................n_neighbors=3; total time=   0.1s
[CV] END ......................................n_neighbors=3; total time=   0.2s
[CV] END .....................................n_neighbors=10; total time=   0.2s
[CV] END .....................................n_neighbors=10; total time=   0.2s
[CV] END .....................................n_neighbors=10; total time=   0.2s
[CV] END .....................................n_neighbors=20; total time=   0.2s
[CV] END .....................................n_neighbors=20; total time=   0.5s
[CV] END .....................................n_n

In [ ]:
models = {
 
        "Random Forest " : RandomForestRegressor(n_estimators=1000, min_samples_split=2,
                                                max_features=7, max_depth=None ),
        "K-Neighbours Regression" : KNeighborsRegressor(n_neighbors=10, n_jobs=-1),

        "Adaboost Regression" : AdaBoostRegressor(n_estimators= , loss=   , )

}

for i in range(len(list(models))): 
    model = list(models.values())[i]
    model.fit(X_train, y_train) # Train Model

    # Make Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    #Evaluate Train and Test Dataset
    model_train_mae , model_train_rmse, model_train_r2 = evaluate_model(y_train, y_train_pred)

    model_test_mae ,  model_test_rmse , model_test_r2 = evaluate_model(y_test, y_test_pred)

    print(list(models.keys())[i])


    # Model Performance for Training Set

    print('Model Performance for Training Set')
    print(" - Root Mean Squared Error : {:.4f}" .format(model_train_rmse))
    print(" - Mean Absolute Error: {:.4f}" .format(model_train_mae))
    print(" - R2 Score : {:.4f}" .format(model_train_r2))

    print("----------------------------------------------------------")

    # Model Performance for Test Set
    
    print('Model Performance for Test Set')
    print(" - Root Mean Squared Error : {:.4f}" .format(model_test_rmse))
    print(" - Mean Absolute Error: {:.4f}" .format(model_test_mae))
    print(" - R2 Score : {:.4f}" .format(model_test_r2))

    print('=' *35)
    print('\n')


Random Forest 
Model Performance for Training Set
 - Root Mean Squared Error : 131610.9630
 - Mean Absolute Error: 38946.1451
 - R2 Score : 0.9786
----------------------------------------------------------
Model Performance for Test Set
 - Root Mean Squared Error : 212396.2907
 - Mean Absolute Error: 98128.0987
 - R2 Score : 0.9401


K-Neighbours Regression
Model Performance for Training Set
 - Root Mean Squared Error : 363460.7006
 - Mean Absolute Error: 103471.6175
 - R2 Score : 0.8371
----------------------------------------------------------
Model Performance for Test Set
 - Root Mean Squared Error : 263882.2582
 - Mean Absolute Error: 117487.1310
 - R2 Score : 0.9075


